# 18_rider_inference_colab.ipynb

**Repository:** pulmonary_nodule_verification v1.0.0 · **Date:** 2026-10-26 · **Developer:** Prof. Wesley Pacheco Calixto · **License:** MIT · **Corresponding author:** Wesley Pacheco Calixto, wesley.compute-node

**Citation:** Calixto, W. P., Camargo, T. F. O., Pacheco, V. M. G., Martins, M. S. R., & Coimbra, A. P. (2026). Multiscale morphology-informed post-detection verification of pulmonary nodule candidates in CT: code, trained nnU-Net models, cohort splits and analysis data (v1.0.0) [Data set and software]. Zenodo. https://doi.org/10.5281/zenodo.23674673

Outputs have been cleared; machine-specific paths were neutralised. See docs/file_level_documentation.md.


# RIDER Lung CT — Inferência nnU-Net (7 modelos) + Matching IoU

Todos os ficheiros ficam no Drive — se a sessão cair, retoma de onde parou.

**Instruções:**
1. Runtime → Change runtime type → **CPU + High RAM**
2. Executar **Célula A** → aguardar `CONCLUÍDO` → **Runtime → Restart session**
3. Executar **Célula B** → verificar `✅` em todos os paths
4. Executar **Células C → N** em sequência

**Se a sessão cair:** re-executar B → continuar da célula onde estava. As predições já concluídas ficam no Drive e são saltadas (`overwrite=False`).

**Ordem das células:**
| Célula | Descrição |
|---|---|
| A | Instalar dependências (requer restart) |
| B | Montar Drive + definir paths + helpers |
| C | Descarregar CTs RIDER do TCIA |
| D | Converter DICOM → NIfTI |
| E–M | Inferência (um modelo por célula) |
| N | Matching IoU (Hungarian) → CSVs |
| O | Verificar resultados no Drive |
| P | Baixar CSVs para Mac |
| Q | Baixar predições para Mac |
| R | Baixar CTs NIfTI para Mac |
| S | Baixar LNDb do Zenodo para Drive |
| T | Código Terminal Mac (descompactar) |

In [ ]:
!pip install -U pip 'setuptools<82' wheel -q
!pip install --no-cache-dir 'numpy==1.26.4' 'nnunetv2==2.7.0' SimpleITK 'scipy>=1.14.0' tcia_utils rarfile requests -q
print('CONCLUÍDO — faça Runtime → Restart session')

In [ ]:
from google.colab import drive
import os, glob, rarfile, torch

if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print('Drive já montado.')

PULMONARY      = '/content/drive/MyDrive/Pulmonary'
RIDER_BASE     = f'{PULMONARY}/RIDER'
RAR_ALL        = f'{PULMONARY}/checkpoint_pth.rar'
RAR_FREM       = f'{PULMONARY}/nnUNetTrainer__nnUNetResEncUNetMPlans__3d_fullres.rar'
NNUNET_RESULTS = '/content/nnunet_results'

INPUT_DIR   = f'{RIDER_BASE}/lung_masks'
PRED_BASE   = f'{RIDER_BASE}/predictions'
RESULTS_DIR = f'{RIDER_BASE}/results'

for d in [RIDER_BASE, INPUT_DIR, PRED_BASE, RESULTS_DIR,
          NNUNET_RESULTS, f'{RIDER_BASE}/nnunet_preprocessed']:
    os.makedirs(d, exist_ok=True)

os.environ.update({
    'nnUNet_raw':          RIDER_BASE,
    'nnUNet_preprocessed': f'{RIDER_BASE}/nnunet_preprocessed',
    'nnUNet_results':      NNUNET_RESULTS,
})

!apt-get install -y unrar -q

def setup_model(model_key, rar_path, nnunet_results, pacheco_prefix=True):
    """Extrai modelo do RAR e devolve (model_dir, checkpoint_name)."""
    model_dir = (f'{nnunet_results}/Pacheco/{model_key}'
                 if pacheco_prefix
                 else os.path.join(nnunet_results, model_key))
    os.makedirs(model_dir, exist_ok=True)

    for ckpt in ['checkpoint_best.pth', 'checkpoint_final.pth']:
        if sorted(glob.glob(f'{model_dir}/**/{ckpt}', recursive=True)):
            print(f'  Checkpoints presentes ({ckpt})')
            return model_dir, ckpt

    print(f'  Extraindo {model_key}...')
    with rarfile.RarFile(rar_path) as rf:
        if pacheco_prefix:
            prefix = f'Pacheco/{model_key}/'
            members = [m for m in rf.infolist() if m.filename.startswith(prefix)]
            print(f'  {len(members)} ficheiros...')
            for m in members:
                rf.extract(m, nnunet_results)
        else:
            members = [m for m in rf.namelist() if model_key in m]
            print(f'  {len(members)} ficheiros...')
            rf.extractall(nnunet_results)

    for ckpt in ['checkpoint_best.pth', 'checkpoint_final.pth']:
        found = sorted(glob.glob(f'{model_dir}/**/{ckpt}', recursive=True))
        if found:
            print(f'  {len(found)} folds com {ckpt}')
            return model_dir, ckpt

    raise FileNotFoundError(f'Checkpoint não encontrado em {model_dir}')

print()
for label, path in [('checkpoint_pth.rar', RAR_ALL), ('RAR 3D_FREM', RAR_FREM)]:
    print('✅' if os.path.exists(path) else '❌', label)
print(f'CTs convertidos: {len(glob.glob(INPUT_DIR+"/*.nii.gz"))}')

---
## Fase 1 — Descarregar e preparar CTs RIDER

In [ ]:
import pandas as pd
from tcia_utils import nbia

DICOM_DIR = f'{RIDER_BASE}/dicom'
os.makedirs(DICOM_DIR, exist_ok=True)

n_nifti = len(glob.glob(f'{INPUT_DIR}/*.nii.gz'))
if n_nifti >= 62:
    print(f'[SKIP] {n_nifti} CTs já convertidos.')
else:
    print('Consultando TCIA...')
    series = nbia.getSeries(collection='RIDER Lung CT', modality='CT')
    df = pd.DataFrame(series)
    pid_col = 'PatientID' if 'PatientID' in df.columns else 'Subject ID'
    uid_col = 'SeriesInstanceUID' if 'SeriesInstanceUID' in df.columns else 'Series UID'
    num_col = 'SeriesNumber' if 'SeriesNumber' in df.columns else 'Series Number'
    bp_col  = 'BodyPartExamined' if 'BodyPartExamined' in df.columns else 'Body Part Examined'
    df = df[df[bp_col] == 'CHEST'].sort_values([pid_col, num_col])
    counts = df.groupby(pid_col).size()
    valid  = counts[counts == 2].index
    df = df[df[pid_col].isin(valid)]
    print(f'{len(valid)} pacientes com 2 séries CT CHEST')
    df.to_csv(f'{RIDER_BASE}/series_metadata.csv', index=False)
    uids = df[uid_col].tolist()
    print(f'Descarregando {len(uids)} séries...')
    nbia.downloadSeries(uids, path=DICOM_DIR, input_type='list')
    print('Download completo.')

In [ ]:
import pandas as pd
import SimpleITK as sitk

n_done = len(glob.glob(f'{INPUT_DIR}/*.nii.gz'))
if n_done >= 62:
    print(f'[SKIP] {n_done} NIfTIs já existem.')
else:
    df = pd.read_csv(f'{RIDER_BASE}/series_metadata.csv')
    pid_col = 'PatientID' if 'PatientID' in df.columns else 'Subject ID'
    uid_col = 'SeriesInstanceUID' if 'SeriesInstanceUID' in df.columns else 'Series UID'
    num_col = 'SeriesNumber' if 'SeriesNumber' in df.columns else 'Series Number'
    df = df.sort_values([pid_col, num_col])
    reader = sitk.ImageSeriesReader()
    errors = []
    for pid, grp in df.groupby(pid_col):
        for acq_idx, sid in enumerate(grp[uid_col].tolist()[:2], start=1):
            out = f'{INPUT_DIR}/{pid}_{acq_idx}_0000.nii.gz'
            if os.path.exists(out):
                continue
            dicom_f = os.path.join(DICOM_DIR, sid)
            if not os.path.isdir(dicom_f):
                candidates = glob.glob(f'{DICOM_DIR}/**/{sid}', recursive=True)
                dicom_f = candidates[0] if candidates else None
            if not dicom_f:
                errors.append((pid, acq_idx))
                continue
            try:
                files = sitk.ImageSeriesReader.GetGDCMSeriesFileNames(dicom_f)
                reader.SetFileNames(files)
                sitk.WriteImage(reader.Execute(), out)
                print(f'  OK {pid} CT{acq_idx}')
            except Exception as e:
                errors.append((pid, acq_idx, str(e)))
    n = len(glob.glob(f'{INPUT_DIR}/*.nii.gz'))
    print(f'Total: {n} NIfTIs. Erros: {len(errors)}')

---
## Fase 2 — Inferência nnU-Net (7 modelos)

Cada célula é independente — se a sessão cair, retome a partir da célula que estava a correr.

In [ ]:
import time
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor

MODEL_KEY, MODEL_LABEL = 'nnUNetTrainer__nnUNetPlans__2d', '2d'
OUTPUT_DIR = f'{PRED_BASE}/{MODEL_LABEL}'
os.makedirs(OUTPUT_DIR, exist_ok=True)
done = len(glob.glob(f'{OUTPUT_DIR}/*.nii.gz'))
need = len(glob.glob(f'{INPUT_DIR}/*.nii.gz'))
print(f'{MODEL_LABEL}: {done}/{need} já no Drive')
if done < need:
    MODEL_DIR, CKPT = setup_model(MODEL_KEY, RAR_ALL, NNUNET_RESULTS)
    predictor = nnUNetPredictor(tile_step_size=0.5, use_gaussian=True, use_mirroring=True,
        perform_everything_on_device=True, device=torch.device('cuda'), verbose=False)
    predictor.initialize_from_trained_model_folder(MODEL_DIR, use_folds=(0,1,2,3,4), checkpoint_name=CKPT)
    t0 = time.time()
    predictor.predict_from_files(INPUT_DIR, OUTPUT_DIR, save_probabilities=False, overwrite=False,
        num_processes_preprocessing=2, num_processes_segmentation_export=2)
    print(f'Concluído em {(time.time()-t0)/60:.1f} min | {len(glob.glob(OUTPUT_DIR+"/*.nii.gz"))} predições')

In [ ]:
import time
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor

MODEL_KEY, MODEL_LABEL = 'nnUNetTrainer__nnUNetResEncUNetMPlans__2d', '2d_resencM'
OUTPUT_DIR = f'{PRED_BASE}/{MODEL_LABEL}'
os.makedirs(OUTPUT_DIR, exist_ok=True)
done = len(glob.glob(f'{OUTPUT_DIR}/*.nii.gz'))
need = len(glob.glob(f'{INPUT_DIR}/*.nii.gz'))
print(f'{MODEL_LABEL}: {done}/{need} já no Drive')
if done < need:
    MODEL_DIR, CKPT = setup_model(MODEL_KEY, RAR_ALL, NNUNET_RESULTS)
    predictor = nnUNetPredictor(tile_step_size=0.5, use_gaussian=True, use_mirroring=True,
        perform_everything_on_device=True, device=torch.device('cuda'), verbose=False)
    predictor.initialize_from_trained_model_folder(MODEL_DIR, use_folds=(0,1,2,3,4), checkpoint_name=CKPT)
    t0 = time.time()
    predictor.predict_from_files(INPUT_DIR, OUTPUT_DIR, save_probabilities=False, overwrite=False,
        num_processes_preprocessing=2, num_processes_segmentation_export=2)
    print(f'Concluído em {(time.time()-t0)/60:.1f} min | {len(glob.glob(OUTPUT_DIR+"/*.nii.gz"))} predições')

In [ ]:
import time
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor

MODEL_KEY, MODEL_LABEL = 'nnUNetTrainer__nnUNetResEncUNetLPlans__2d', '2d_resencL'
OUTPUT_DIR = f'{PRED_BASE}/{MODEL_LABEL}'
os.makedirs(OUTPUT_DIR, exist_ok=True)
done = len(glob.glob(f'{OUTPUT_DIR}/*.nii.gz'))
need = len(glob.glob(f'{INPUT_DIR}/*.nii.gz'))
print(f'{MODEL_LABEL}: {done}/{need} já no Drive')
if done < need:
    MODEL_DIR, CKPT = setup_model(MODEL_KEY, RAR_ALL, NNUNET_RESULTS)
    predictor = nnUNetPredictor(tile_step_size=0.5, use_gaussian=True, use_mirroring=True,
        perform_everything_on_device=True, device=torch.device('cuda'), verbose=False)
    predictor.initialize_from_trained_model_folder(MODEL_DIR, use_folds=(0,1,2,3,4), checkpoint_name=CKPT)
    t0 = time.time()
    predictor.predict_from_files(INPUT_DIR, OUTPUT_DIR, save_probabilities=False, overwrite=False,
        num_processes_preprocessing=2, num_processes_segmentation_export=2)
    print(f'Concluído em {(time.time()-t0)/60:.1f} min | {len(glob.glob(OUTPUT_DIR+"/*.nii.gz"))} predições')

In [ ]:
import time
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor

MODEL_KEY, MODEL_LABEL = 'nnUNetTrainer__nnUNetPlans__3d_fullres', '3d_fullres'
OUTPUT_DIR = f'{PRED_BASE}/{MODEL_LABEL}'
os.makedirs(OUTPUT_DIR, exist_ok=True)
done = len(glob.glob(f'{OUTPUT_DIR}/*.nii.gz'))
need = len(glob.glob(f'{INPUT_DIR}/*.nii.gz'))
print(f'{MODEL_LABEL}: {done}/{need} já no Drive')
if done < need:
    MODEL_DIR, CKPT = setup_model(MODEL_KEY, RAR_ALL, NNUNET_RESULTS)
    predictor = nnUNetPredictor(tile_step_size=0.5, use_gaussian=True, use_mirroring=True,
        perform_everything_on_device=True, device=torch.device('cuda'), verbose=False)
    predictor.initialize_from_trained_model_folder(MODEL_DIR, use_folds=(0,1,2,3,4), checkpoint_name=CKPT)
    t0 = time.time()
    predictor.predict_from_files(INPUT_DIR, OUTPUT_DIR, save_probabilities=False, overwrite=False,
        num_processes_preprocessing=2, num_processes_segmentation_export=2)
    print(f'Concluído em {(time.time()-t0)/60:.1f} min | {len(glob.glob(OUTPUT_DIR+"/*.nii.gz"))} predições')

In [ ]:
import time
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor

MODEL_KEY, MODEL_LABEL = 'nnUNetTrainer__nnUNetResEncUNetMPlans__3d_fullres', '3d_fullres_resencM'
OUTPUT_DIR = f'{PRED_BASE}/{MODEL_LABEL}'
os.makedirs(OUTPUT_DIR, exist_ok=True)
done = len(glob.glob(f'{OUTPUT_DIR}/*.nii.gz'))
need = len(glob.glob(f'{INPUT_DIR}/*.nii.gz'))
print(f'{MODEL_LABEL}: {done}/{need} já no Drive')
if done < need:
    rar = RAR_FREM if os.path.exists(RAR_FREM) else RAR_ALL
    MODEL_DIR, CKPT = setup_model(MODEL_KEY, rar, NNUNET_RESULTS, pacheco_prefix=False)
    predictor = nnUNetPredictor(tile_step_size=0.5, use_gaussian=True, use_mirroring=True,
        perform_everything_on_device=True, device=torch.device('cuda'), verbose=False)
    predictor.initialize_from_trained_model_folder(MODEL_DIR, use_folds=(0,1,2,3,4), checkpoint_name=CKPT)
    t0 = time.time()
    predictor.predict_from_files(INPUT_DIR, OUTPUT_DIR, save_probabilities=False, overwrite=False,
        num_processes_preprocessing=2, num_processes_segmentation_export=2)
    print(f'Concluído em {(time.time()-t0)/60:.1f} min | {len(glob.glob(OUTPUT_DIR+"/*.nii.gz"))} predições')

In [ ]:
import time
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor

MODEL_KEY, MODEL_LABEL = 'nnUNetTrainer__nnUNetPlans__3d_lowres', '_3d_lowres'
OUTPUT_DIR = f'{PRED_BASE}/{MODEL_LABEL}'
os.makedirs(OUTPUT_DIR, exist_ok=True)
done = len(glob.glob(f'{OUTPUT_DIR}/*.nii.gz'))
need = len(glob.glob(f'{INPUT_DIR}/*.nii.gz'))
print(f'{MODEL_LABEL}: {done}/{need} já no Drive')
if done < need:
    MODEL_DIR, CKPT = setup_model(MODEL_KEY, RAR_ALL, NNUNET_RESULTS)
    predictor = nnUNetPredictor(tile_step_size=0.5, use_gaussian=True, use_mirroring=True,
        perform_everything_on_device=True, device=torch.device('cuda'), verbose=False)
    predictor.initialize_from_trained_model_folder(MODEL_DIR, use_folds=(0,1,2,3,4), checkpoint_name=CKPT)
    t0 = time.time()
    predictor.predict_from_files(INPUT_DIR, OUTPUT_DIR, save_probabilities=False, overwrite=False,
        num_processes_preprocessing=2, num_processes_segmentation_export=2)
    print(f'Concluído em {(time.time()-t0)/60:.1f} min | {len(glob.glob(OUTPUT_DIR+"/*.nii.gz"))} predições')
else:
    print('  → já concluído.')

In [ ]:
import time
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor

MODEL_KEY, MODEL_LABEL = 'nnUNetTrainer__nnUNetPlans__3d_cascade_fullres', '3d_cascade_fullres'
OUTPUT_DIR = f'{PRED_BASE}/{MODEL_LABEL}'
LOWRES_DIR = f'{PRED_BASE}/_3d_lowres'
os.makedirs(OUTPUT_DIR, exist_ok=True)
done   = len(glob.glob(f'{OUTPUT_DIR}/*.nii.gz'))
need   = len(glob.glob(f'{INPUT_DIR}/*.nii.gz'))
lowres = len(glob.glob(f'{LOWRES_DIR}/*.nii.gz'))
print(f'{MODEL_LABEL}: {done}/{need} | lowres: {lowres}/{need}')
assert lowres >= need, f'Execute Célula J primeiro! ({lowres}/{need})'
if done < need:
    MODEL_DIR, CKPT = setup_model(MODEL_KEY, RAR_ALL, NNUNET_RESULTS)
    predictor = nnUNetPredictor(tile_step_size=0.5, use_gaussian=True, use_mirroring=True,
        perform_everything_on_device=True, device=torch.device('cuda'), verbose=False)
    predictor.initialize_from_trained_model_folder(MODEL_DIR, use_folds=(0,1,2,3,4), checkpoint_name=CKPT)
    t0 = time.time()
    predictor.predict_from_files(INPUT_DIR, OUTPUT_DIR, save_probabilities=False, overwrite=False,
        num_processes_preprocessing=2, num_processes_segmentation_export=2,
        folder_with_segs_from_prev_stage=LOWRES_DIR)
    print(f'Concluído em {(time.time()-t0)/60:.1f} min | {len(glob.glob(OUTPUT_DIR+"/*.nii.gz"))} predições')

In [ ]:
import time
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor

MODEL_KEY, MODEL_LABEL = 'nnUNetTrainer__nnUNetResEncUNetMPlans__3d_lowres', '_3d_lowres_resencM'
OUTPUT_DIR = f'{PRED_BASE}/{MODEL_LABEL}'
os.makedirs(OUTPUT_DIR, exist_ok=True)
done = len(glob.glob(f'{OUTPUT_DIR}/*.nii.gz'))
need = len(glob.glob(f'{INPUT_DIR}/*.nii.gz'))
print(f'{MODEL_LABEL}: {done}/{need} já no Drive')
if done < need:
    MODEL_DIR, CKPT = setup_model(MODEL_KEY, RAR_ALL, NNUNET_RESULTS)
    predictor = nnUNetPredictor(tile_step_size=0.5, use_gaussian=True, use_mirroring=True,
        perform_everything_on_device=True, device=torch.device('cuda'), verbose=False)
    predictor.initialize_from_trained_model_folder(MODEL_DIR, use_folds=(0,1,2,3,4), checkpoint_name=CKPT)
    t0 = time.time()
    predictor.predict_from_files(INPUT_DIR, OUTPUT_DIR, save_probabilities=False, overwrite=False,
        num_processes_preprocessing=2, num_processes_segmentation_export=2)
    print(f'Concluído em {(time.time()-t0)/60:.1f} min | {len(glob.glob(OUTPUT_DIR+"/*.nii.gz"))} predições')
else:
    print('  → já concluído.')

In [ ]:
import time
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor

MODEL_KEY   = 'nnUNetTrainer__nnUNetResEncUNetMPlans__3d_cascade_fullres'
MODEL_LABEL = '3d_cascade_fullres_resencM'
OUTPUT_DIR  = f'{PRED_BASE}/{MODEL_LABEL}'
LOWRES_DIR  = f'{PRED_BASE}/_3d_lowres_resencM'
os.makedirs(OUTPUT_DIR, exist_ok=True)
done   = len(glob.glob(f'{OUTPUT_DIR}/*.nii.gz'))
need   = len(glob.glob(f'{INPUT_DIR}/*.nii.gz'))
lowres = len(glob.glob(f'{LOWRES_DIR}/*.nii.gz'))
print(f'{MODEL_LABEL}: {done}/{need} | lowres: {lowres}/{need}')
assert lowres >= need, f'Execute Célula L primeiro! ({lowres}/{need})'
if done < need:
    MODEL_DIR, CKPT = setup_model(MODEL_KEY, RAR_ALL, NNUNET_RESULTS)
    predictor = nnUNetPredictor(tile_step_size=0.5, use_gaussian=True, use_mirroring=True,
        perform_everything_on_device=True, device=torch.device('cuda'), verbose=False)
    predictor.initialize_from_trained_model_folder(MODEL_DIR, use_folds=(0,1,2,3,4), checkpoint_name=CKPT)
    t0 = time.time()
    predictor.predict_from_files(INPUT_DIR, OUTPUT_DIR, save_probabilities=False, overwrite=False,
        num_processes_preprocessing=2, num_processes_segmentation_export=2,
        folder_with_segs_from_prev_stage=LOWRES_DIR)
    print(f'Concluído em {(time.time()-t0)/60:.1f} min | {len(glob.glob(OUTPUT_DIR+"/*.nii.gz"))} predições')

---
## Fase 3 — Matching IoU (Hungarian)

In [ ]:
import numpy as np
import pandas as pd
import SimpleITK as sitk
from scipy.ndimage import label as nd_label
from scipy.optimize import linear_sum_assignment

def _sphere_vol(r): return (4/3) * 3.14159 * r**3
MIN_VOL = _sphere_vol(1.5)
MAX_VOL = _sphere_vol(15.0)

def filter_mask(m):
    a  = sitk.GetArrayFromImage(m).astype(bool)
    vv = np.prod(m.GetSpacing())
    lbl, n = nd_label(a, structure=np.ones((3,3,3), int))
    sz = np.bincount(lbl.ravel()) * vv; sz[0] = 0
    out = np.zeros_like(a)
    for i in range(1, n+1):
        if MIN_VOL <= sz[i] <= MAX_VOL:
            out[lbl==i] = True
    r = sitk.GetImageFromArray(out.astype(np.uint8))
    r.CopyInformation(m)
    return r

def register(fix, mov):
    reg = sitk.ImageRegistrationMethod()
    reg.SetMetricAsMeanSquares()
    reg.SetOptimizerAsRegularStepGradientDescent(1.0, 1e-6, 200, 1e-6)
    reg.SetOptimizerScalesFromPhysicalShift()
    reg.SetInterpolator(sitk.sitkLinear)
    tx = sitk.CenteredTransformInitializer(
        fix, mov, sitk.Euler3DTransform(),
        sitk.CenteredTransformInitializerFilter.GEOMETRY)
    reg.SetInitialTransform(tx, inPlace=False)
    return reg.Execute(fix, mov)

def label_comps(m):
    a  = sitk.GetArrayFromImage(m)
    vv = np.prod(m.GetSpacing())
    lbl, n = nd_label(a == 1, structure=np.ones((3,3,3), int))
    return lbl, {i: int((lbl==i).sum()) * vv for i in range(1, n+1)}

def bbox_diam(lbl, lid, sp):
    c = np.argwhere(lbl == lid)
    if not c.size: return 0.0
    mx = 0.0
    for z in np.unique(c[:,0]):
        s = c[c[:,0]==z]
        mx = max(mx,
                 (s[:,1].max() - s[:,1].min() + 1) * sp[1],
                 (s[:,2].max() - s[:,2].min() + 1) * sp[0])
    return mx

MODEL_MAP = {
    '2d':                         '2D',
    '2d_resencM':                 '2D_REM',
    '2d_resencL':                 '2D_REL',
    '3d_fullres':                 '3D_F',
    '3d_fullres_resencM':         '3D_FREM',
    '3d_cascade_fullres':         '3D_CF',
    '3d_cascade_fullres_resencM': '3D_CFREM',
}

all_vol, all_diam, all_counts = [], [], []

for model_dir, model_label in MODEL_MAP.items():
    mpath = f'{PRED_BASE}/{model_dir}'
    if not os.path.isdir(mpath):
        print(f'[SKIP] {model_label}')
        continue
    pids = sorted({f.replace('_1.nii.gz','').replace('_2.nii.gz','')
                   for f in os.listdir(mpath) if f.endswith('.nii.gz')})
    print(f'\n=== {model_label} ({len(pids)} pacientes) ===')
    for pid in pids:
        m1p = f'{mpath}/{pid}_1.nii.gz'
        m2p = f'{mpath}/{pid}_2.nii.gz'
        l1p = f'{INPUT_DIR}/{pid}_1_0000.nii.gz'
        l2p = f'{INPUT_DIR}/{pid}_2_0000.nii.gz'
        if not all(os.path.exists(p) for p in [m1p, m2p, l1p, l2p]):
            print(f'  [FALTA] {pid}')
            continue
        lm1 = sitk.Cast(sitk.ReadImage(l1p, sitk.sitkUInt8), sitk.sitkFloat32)
        lm2 = sitk.Cast(sitk.ReadImage(l2p, sitk.sitkUInt8), sitk.sitkFloat32)
        tx  = register(lm1, lm2)
        m1  = sitk.ReadImage(m1p, sitk.sitkUInt8)
        m2  = sitk.ReadImage(m2p, sitk.sitkUInt8)
        m2r = sitk.Resample(m2, m1, tx, sitk.sitkNearestNeighbor, 0, m2.GetPixelID())
        lbl1, v1 = label_comps(filter_mask(m1))
        lbl2, v2 = label_comps(filter_mask(m2r))
        ls1, ls2 = sorted(v1), sorted(v2)
        if not ls1 or not ls2:
            print(f'  {pid}: 0 pares (CT1:{len(ls1)} CT2:{len(ls2)})')
            all_counts.append({'patient_id': pid, 'model': model_label,
                               'n_pairs_hung': 0, 'ct1_only': len(ls1), 'ct2_only': len(ls2)})
            continue
        iou = np.zeros((len(ls1), len(ls2)))
        for i, l1 in enumerate(ls1):
            b1 = (lbl1 == l1)
            for j, l2 in enumerate(ls2):
                inter = np.logical_and(b1, lbl2 == l2).sum()
                if inter:
                    iou[i,j] = inter / np.logical_or(b1, lbl2 == l2).sum()
        rows, cols = linear_sum_assignment(-iou)
        pairs = [(ls1[r], ls2[c], float(iou[r,c])) for r,c in zip(rows,cols) if iou[r,c] > 0]
        sp  = m1.GetSpacing()
        mt1 = {l for l,_,_ in pairs}
        mt2 = {l for _,l,_ in pairs}
        for l1, l2, iov in pairs:
            d1 = bbox_diam(lbl1, l1, sp)
            d2 = bbox_diam(lbl2, l2, sp)
            all_vol.append({'patient_id': pid, 'model': model_label,
                            'label1': l1, 'label2': l2,
                            'vol_ct1': v1[l1], 'vol_ct2': v2[l2],
                            'vol_diff': v2[l2]-v1[l1], 'iou': iov})
            all_diam.append({'patient_id': pid, 'model': model_label,
                             'label1': l1, 'label2': l2,
                             'diam_ct1': d1, 'diam_ct2': d2, 'diam_diff': d2-d1})
        all_counts.append({'patient_id': pid, 'model': model_label,
                           'n_pairs_hung': len(pairs),
                           'ct1_only': len(set(ls1)-mt1),
                           'ct2_only': len(set(ls2)-mt2)})
        print(f'  {pid}: {len(pairs)} pares (CT1:{len(ls1)} CT2:{len(ls2)})')

pd.DataFrame(all_vol).to_csv(f'{RESULTS_DIR}/volumes_RIDER_iou.csv', index=False)
pd.DataFrame(all_diam).to_csv(f'{RESULTS_DIR}/diametros_RIDER_iou.csv', index=False)
pd.DataFrame(all_counts).to_csv(f'{RESULTS_DIR}/rider_pair_counts_iou.csv', index=False)
print('\nResumo por modelo:')
print(pd.DataFrame(all_counts).groupby('model')[['n_pairs_hung','ct1_only','ct2_only']].sum())
print(f'CSVs guardados em {RESULTS_DIR}')

---
## Fase 4 — Download dos resultados para o Mac

In [ ]:
print('=== results/ ===')
for f in sorted(os.listdir(RESULTS_DIR)):
    sz = os.path.getsize(os.path.join(RESULTS_DIR, f))
    print(f'  {f}  ({sz/1024:.1f} KB)')

print('\n=== predictions/ (contagem por modelo) ===')
for model in sorted(os.listdir(PRED_DIR)):
    mp = os.path.join(PRED_DIR, model)
    if os.path.isdir(mp):
        n = len([f for f in os.listdir(mp) if f.endswith('.nii.gz')])
        print(f'  {model}: {n} arquivos')

print(f'\n=== lung_masks/ ===')
n = len([f for f in os.listdir(INPUT_DIR) if f.endswith('.nii.gz')])
print(f'  {n} NIfTIs')

In [ ]:
import shutil
from google.colab import files

LOCAL_TMP = '/content/rider_csvs'
os.makedirs(LOCAL_TMP, exist_ok=True)
for csv in ['volumes_RIDER_iou.csv', 'diametros_RIDER_iou.csv', 'rider_pair_counts_iou.csv']:
    src = os.path.join(RESULTS_DIR, csv)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(LOCAL_TMP, csv))
        print(f'OK  {csv}')
    else:
        print(f'FALTA  {csv}')

zip_path = '/content/RIDER_csvs.zip'
shutil.make_archive('/content/RIDER_csvs', 'zip', LOCAL_TMP)
print(f'Baixando RIDER_csvs.zip ({os.path.getsize(zip_path)/1024:.1f} KB)...')
files.download(zip_path)

In [ ]:
import shutil, glob
from google.colab import files

MODELOS = ['2d', '2d_resencM', '2d_resencL', '3d_fullres',
           '3d_fullres_resencM', '3d_cascade_fullres', '3d_cascade_fullres_resencM']

for model in MODELOS:
    src = os.path.join(PRED_BASE, model)
    if not os.path.isdir(src):
        print(f'[SKIP] {model} — não encontrado')
        continue
    n = len(glob.glob(f'{src}/*.nii.gz'))
    if n == 0:
        print(f'[SKIP] {model} — 0 predições')
        continue
    zip_path = f'/content/RIDER_pred_{model}.zip'
    print(f'Compactando {model} ({n} arquivos)...')
    shutil.make_archive(f'/content/RIDER_pred_{model}', 'zip', PRED_BASE, model)
    sz_mb = os.path.getsize(zip_path) / 1024 / 1024
    print(f'  Baixando ({sz_mb:.1f} MB)...')
    files.download(zip_path)

print('Concluído.')

In [ ]:
import shutil
from google.colab import files

n = len([f for f in os.listdir(INPUT_DIR) if f.endswith('.nii.gz')])
print(f'Compactando {n} NIfTIs...')
shutil.make_archive('/content/RIDER_lung_masks', 'zip', INPUT_DIR)
zip_path = '/content/RIDER_lung_masks.zip'
sz_mb = os.path.getsize(zip_path) / 1024 / 1024
print(f'Baixando RIDER_lung_masks.zip ({sz_mb:.1f} MB)...')
files.download(zip_path)

---
## Fase 5 — Download LNDb (Zenodo → Drive)

In [ ]:
import json, requests, subprocess

LNDB_DIR   = f'{PULMONARY}/LNDb'
ZENODO_URL = 'https://zenodo.org/api/records/8348419'
os.makedirs(LNDB_DIR, exist_ok=True)

subprocess.run(['apt-get', 'install', '-y', 'unrar'], capture_output=True)

meta_cache = f'{LNDB_DIR}/zenodo_meta.json'
if os.path.isfile(meta_cache):
    with open(meta_cache) as f:
        meta = json.load(f)
else:
    print('Consultando Zenodo...')
    meta = requests.get(ZENODO_URL, timeout=60).json()
    with open(meta_cache, 'w') as f:
        json.dump(meta, f)

files_info = meta.get('files', [])
doi        = meta.get('doi', '')
total_gb   = sum(fi.get('size', 0) for fi in files_info) / 1e9
print(f'LNDb | DOI: {doi} | {len(files_info)} arquivo(s) | {total_gb:.1f} GB total')

for fi in files_info:
    fname    = fi['key']
    url      = fi['links']['self']
    dest     = os.path.join(LNDB_DIR, fname)
    tmp      = dest + '.tmp'
    size_mb  = fi.get('size', 0) / 1e6
    if os.path.isfile(dest):
        print(f'  [OK] {fname} ({size_mb:.0f} MB)')
        continue
    headers    = {}
    downloaded = 0
    if os.path.isfile(tmp):
        downloaded = os.path.getsize(tmp)
        headers['Range'] = f'bytes={downloaded}-'
        print(f'  Retomando {fname} a partir de {downloaded/1e6:.0f} MB...')
    else:
        print(f'  Baixando {fname} ({size_mb:.0f} MB)...')
    r = requests.get(url, stream=True, timeout=1800, headers=headers)
    if r.status_code == 416:
        os.rename(tmp, dest)
    else:
        r.raise_for_status()
        mode = 'ab' if downloaded > 0 else 'wb'
        with open(tmp, mode) as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
        os.rename(tmp, dest)
    print(f'  Concluído: {fname}')
    if fname.endswith('.rar'):
        print(f'  Extraindo {fname}...')
        subprocess.run(['unrar', 'x', '-o+', dest, LNDB_DIR], check=True)
        print(f'  Extraído.')

print('\nLNDb concluído.')

In [ ]:
print("""
=== COLAR NO TERMINAL DO MAC ===

BASE="<external>/Arquivos Pacheco 02_05_2022/Programacao/Programa_Pulmonary/Ultima_Versao/Pulmonary_Project"
DL=~/Downloads

# CSVs
mkdir -p "$BASE/figdata" "$BASE/analysis"
unzip -o "$DL/RIDER_csvs.zip" -d /tmp/rider_csvs/
cp /tmp/rider_csvs/volumes_RIDER_iou.csv     "$BASE/figdata/"
cp /tmp/rider_csvs/diametros_RIDER_iou.csv   "$BASE/figdata/"
cp /tmp/rider_csvs/rider_pair_counts_iou.csv "$BASE/analysis/"

# Predições
for model in 2d 2d_resencM 2d_resencL 3d_fullres 3d_fullres_resencM 3d_cascade_fullres 3d_cascade_fullres_resencM; do
  mkdir -p "$BASE/predictions/rider/$model"
  unzip -o "$DL/RIDER_pred_${model}.zip" -d "$BASE/predictions/rider/"
done

# CTs NIfTI
mkdir -p "$BASE/data/processed/rider_nifti"
unzip -o "$DL/RIDER_lung_masks.zip" -d "$BASE/data/processed/rider_nifti/"

echo 'Concluído.'
""")